# Cleaning test (T02)
Kiểm tra kết quả làm sạch. Chạy sau `01_cleaning.sql` và `cleaning_python.ipynb`. Cần `cars_cleaned.csv` cùng thư mục.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

SERVER = "localhost\\TEST"
engine = create_engine(
    f"mssql+pyodbc://@{SERVER}/UsedCars"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes&TrustServerCertificate=yes"
)
raw = pd.read_sql("SELECT * FROM cars_raw", engine)
sql_clean = pd.read_sql("SELECT * FROM cars_clean", engine)
final = pd.read_csv("cars_cleaned.csv", encoding="utf-8-sig")

raw["price_million"] = pd.to_numeric(raw["price"], errors="coerce") / 1e6
raw["odo_km"] = pd.to_numeric(raw["mileage_v2"], errors="coerce")

## 1. Số dòng qua từng bước

In [ ]:
funnel = pd.DataFrame({
    "Bước": ["Dữ liệu thô (cars_raw)", "Sau SQL (cars_clean)", "Sau Python IQR (file cuối)"],
    "Số dòng": [len(raw), len(sql_clean), len(final)],
})
funnel["Đã loại so với bước trước"] = (-funnel["Số dòng"].diff()).fillna(0).astype(int)
funnel

## 2. Kiểm tra quy tắc cleaning (PASS/FAIL)

In [ ]:
results = []
def check(name, cond, detail=""):
    results.append({"Kiểm tra": name, "Kết quả": "PASS" if cond else "FAIL", "Chi tiết": detail})

check("Không trùng list_id", final["list_id"].is_unique, f"{final['list_id'].duplicated().sum()} dòng trùng")
key = ["carbrand_name", "carmodel_name", "mfdate", "odo_km", "region_name", "price_million"]
n_dup = final.duplicated(subset=key).sum()
check("Không trùng xe (hãng, dòng, năm, ODO, tỉnh, giá)", n_dup == 0, f"{n_dup} dòng trùng")
check("Không thiếu giá / năm sản xuất", final[["price_million", "mfdate"]].notna().all().all())
check("Giá >= 20 triệu", (final["price_million"] >= 20).all(), f"min = {final['price_million'].min():.1f}")
check("ODO không thiếu", final["odo_km"].notna().all())
check("ODO > 0", (final["odo_km"] > 0).all(), f"min = {final['odo_km'].min():.0f}")
check("Tuổi xe >= 0", (final["car_age"] >= 0).all(), f"max = {final['car_age'].max():.0f}")

spam = r"cho thuê|thuê xe|tìm mua|cần mua|thu mua|góp xe|đặt cọc"
n_spam = final["subject"].astype(str).str.contains(spam, case=False, regex=True).sum()
check("Không còn tin spam / thuê / tìm mua", n_spam == 0, f"{n_spam} dòng còn sót")

for c in ["flag_chinh_chu", "flag_bao_duong_hang"]:
    check(f"{c} chỉ có 0/1", set(final[c].unique()) <= {0, 1})

for c in ["gearbox", "fuel", "cartype", "carcolor", "carorigin", "carseats"]:
    n_num = final[c].astype(str).str.fullmatch(r"\d+(\.0)?").sum()
    unk = (final[c] == "Unknown").mean()
    check(f"{c} không còn mã số thô", n_num == 0, f"Unknown: {unk:.1%}")

need = ["price_million", "odo_km", "car_age", "carbrand_name", "carmodel_name",
        "gearbox", "fuel", "region_name", "flag_chinh_chu", "flag_bao_duong_hang"]
missing = [c for c in need if c not in final.columns]
check("Đủ cột cho notebook mô hình", not missing, f"thiếu: {missing}")

pd.DataFrame(results)

## 3. Trước và sau cleaning

In [ ]:
compare = pd.DataFrame({
    "Giá thô": raw["price_million"].describe(),
    "Giá sạch": final["price_million"].describe(),
    "ODO thô": raw["odo_km"].describe(),
    "ODO sạch": final["odo_km"].describe(),
}).round(1)
display(compare)

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0, 0].hist(raw["price_million"].dropna().clip(upper=3000), bins=60); ax[0, 0].set_title("Giá thô (triệu, hiển thị cắt ở 3000)")
ax[0, 1].hist(final["price_million"], bins=60); ax[0, 1].set_title("Giá sạch (triệu)")
ax[1, 0].hist(raw["odo_km"].dropna().clip(upper=500000), bins=60); ax[1, 0].set_title("ODO thô (km, hiển thị cắt ở 500k)")
ax[1, 1].hist(final["odo_km"], bins=60); ax[1, 1].set_title("ODO sạch (km)")
plt.tight_layout(); plt.show()

## 4. Kiểm chứng bảng mã carseats bằng dòng xe

In [ ]:
raw["seat_code"] = pd.to_numeric(raw["carseats"], errors="coerce")
for n in [4, 5, 7]:
    m = raw["subject"].astype(str).str.contains(rf"(?<!\d){n} chỗ", case=False, regex=True)
    print(f"Tiêu đề có '{n} chỗ' ({m.sum()} tin) -> mã carseats:", raw[m]["seat_code"].value_counts().head(4).to_dict())

print()
for code in sorted(raw["seat_code"].dropna().unique()):
    models = raw[raw["seat_code"] == code]["carmodel_name"].value_counts().head(4).to_dict()
    print(f"Mã {int(code)}:", models)